In [1]:
import os  # Operating system interface for file/directory operations
import numpy as np  # Numerical computing with arrays and matrices
import pandas as pd  # Data manipulation and analysis library
import matplotlib.pyplot as plt  # 2D plotting library for visualizations
import seaborn as sns  # Statistical data visualization built on matplotlib
import re  # Regular expressions for pattern matching in strings
import nltk  # Natural Language Toolkit for NLP tasks
from nltk.corpus import stopwords  # Common words to filter from text (a, the, is, etc.)
nltk.download('stopwords')  # Download stopwords dataset
import string  # Common string operations and character constants
from sklearn.model_selection import train_test_split  # Split data into training and testing sets
from sklearn.feature_extraction.text import CountVectorizer  # Convert text documents to word count matrix
from sklearn.feature_extraction.text import TfidfTransformer  # Transform word counts to TF-IDF (importance) scores
from tensorflow.keras.models import Model  # Base class for Keras models
from tensorflow.keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding, SpatialDropout1D  # Neural network layer types
from tensorflow.keras.optimizers import RMSprop  # RMSprop optimization algorithm for training
from tensorflow.keras.preprocessing.text import Tokenizer  # Convert text into sequences of integers
from tensorflow.keras.preprocessing import sequence  # Utilities for preprocessing sequences (padding, truncating)
from tensorflow.keras.utils import to_categorical  # Convert class integers to one-hot encoded vectors
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint  # Training callbacks for stopping and saving models


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/muditrustagi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Load the IMDB reviews CSV and preview the first 5 rows
df = pd.read_csv('/Users/muditrustagi/Documents/Study/Generative AI - Comprehensive/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# Rows x columns of the full dataset
df.shape

(50000, 2)

In [4]:
# Keep only the first 10,000 reviews to speed up experiments
df = df.iloc[:10000]
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
# Confirm the shape after sampling
df.shape

(10000, 2)

In [6]:
# Count positive vs negative reviews to check class balance
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [7]:
# Count missing values per column
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
# Count fully duplicated rows
df.duplicated().sum()

np.int64(17)

In [9]:
# Drop duplicate rows in place
df.drop_duplicates(inplace=True)

In [10]:
# Confirm no duplicates remain (should be 0)
df.duplicated().sum()

np.int64(0)

## Basic Preprocessing

Remove HTML tags

Lowercase

Remove Stopwords

In [11]:
# Strip HTML tags (e.g. <br />) from a review using a non-greedy regex
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text
    

In [12]:
# Apply HTML-tag removal to every review
df['review'] = df['review'].apply(remove_tags)

In [13]:
# Inspect cleaned reviews
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [14]:
# Lowercase all text so 'Movie' and 'movie' are treated the same
df['review'] = df['review'].apply(lambda x : x.lower())

In [15]:
# Spot-check the first review after lowercasing
df['review'][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows wo

In [16]:
# Remove English stopwords (the, is, and...) that add little meaning
# split -> filter against stopword list -> join back into a string
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/muditrustagi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
# Spot-check the first review after stopword removal
df['review'][0]

"one reviewers mentioned watching 1 oz episode hooked. right, exactly happened me.the first thing struck oz brutality unflinching scenes violence, set right word go. trust me, show faint hearted timid. show pulls punches regards drugs, sex violence. hardcore, classic use word.it called oz nickname given oswald maximum security state penitentary. focuses mainly emerald city, experimental section prison cells glass fronts face inwards, privacy high agenda. em city home many..aryans, muslims, gangstas, latinos, christians, italians, irish more....so scuffles, death stares, dodgy dealings shady agreements never far away.i would say main appeal show due fact goes shows dare. forget pretty pictures painted mainstream audiences, forget charm, forget romance...oz mess around. first episode ever saw struck nasty surreal, say ready it, watched more, developed taste oz, got accustomed high levels graphic violence. violence, injustice (crooked guards who'll sold nickel, inmates who'll kill order g

In [18]:
# Inspect the preprocessed dataframe
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive


In [19]:
# X = review text (feature), y = sentiment label (target)
X = df.iloc[:,0:1]
y = df['sentiment']

In [20]:
# Preview features
X.head()

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."


In [21]:
# Preview labels
y.head()

0    positive
1    positive
2    positive
3    negative
4    positive
Name: sentiment, dtype: str

In [22]:
# Convert string sentiment labels (positive/negative) into integers (0/1) for the model
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [23]:
# Inspect the encoded labels
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [24]:
# Split into 80% train / 20% test; random_state makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=22)

In [25]:
# Confirm the training set size
X_train.shape

(7986, 1)

In [26]:
# Bag-of-Words vectorizer: turns each review into word-count features
cv = CountVectorizer()

In [27]:
# Learn the vocabulary from training reviews and convert them to dense count vectors
X_train_bow = cv.fit_transform(X_train['review']).toarray()

In [28]:
# Transform test reviews using the training vocabulary (no refitting, avoids data leakage)
X_test_bow = cv.transform(X_test['review']).toarray()

In [29]:
# Inspect the training Bag-of-Words matrix
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(7986, 48099))

In [30]:
# Gaussian Naive Bayes baseline classifier (requires dense input)
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

In [31]:
# Train Naive Bayes on the training vectors
gnb.fit(X_train_bow, y_train)


,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](2,)","[3967.,4019.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](2,)","[0.5,0.5]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](2,)","[0,1]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,5.318e-09
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,48099
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](2, 48099)","[[0. ,0.02,0. ,...,0. ,0. ,0. ], [0. ,0.01,0. ,...,0. ,0. ,0. ]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](2, 48099)","[[0. ,0.03,0. ,...,0. ,0. ,0. ], [0. ,0.01,0. ,...,0. ,0. ,0. ]]"


In [32]:
# Predict on the test set and measure accuracy
from sklearn.metrics import accuracy_score
y_pred = gnb.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.6329494241362043

In [33]:
# Confusion matrix: rows = actual class, columns = predicted class
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)

array([[691, 302],
       [431, 573]])

In [34]:
# Train a Random Forest on the full vocabulary and compare its accuracy with Naive Bayes
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)

y_pred = rf.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.8597896845267902

In [35]:
# Limit the vocabulary to the 3000 most frequent words to cut dimensionality and training time
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()


In [36]:
# Create a fresh Random Forest to train on the reduced feature set
rf = RandomForestClassifier()

In [37]:
# Fit on the 3000-feature data and evaluate accuracy on the test set
rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)

accuracy_score(y_test, y_pred)

0.8417626439659489